# 4. Od wzorców F do książki T–S–F–D

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/04_vibe_coding/04_od_wzorcow_do_ksiazki_TSFD.ipynb)

Cel: zbudować jedną operacyjną kartę S, oszczędny zakres T i jawne przekazanie do Codexa.

Odznaka otwiera publiczny wzorzec. Do trwałej pracy utwórz prywatne
repo przez **Use this template**, dodaj sekrety `AI_QDA_REPOSITORY` i
`GITHUB_TOKEN`, a notebook otwieraj z własnego repo przez
**Plik → Otwórz notatnik → GitHub**.

To jest ćwiczenie z **projektowania pipeline'u kodowania AI_QDA**.
Nie odtwarza autorskiego generatora pełnego wyniku. Kod techniczny jest
zwinięty; widoczne pozostają materiał, karta procedury, odpowiedzi modelu
i decyzja badacza.

W promptach **CZĘŚĆ BADAWCZA** pochodzi z karty uczestnika, a
**DODATEK TECHNICZNY** tylko dopasowuje jedną funkcję do notebooka.


## Rytm pracy

`F i cytaty → operacyjna potrzeba S → porównanie dwóch syntez →
karta S i zakres T → kontrola hierarchii → review → handoff`

`Synthetic coding` oznacza syntezę znaczeń z F, a nie sztuczne dane.


## Dwa porządki pracy — nie mieszamy ich

**1. Tworzenie kodu:** krótką funkcję projektujesz w czacie AI
zintegrowanym z Colabem. Wysyłasz tam instrukcję o procedurze i
kontrakcie funkcji, a otrzymany kod wklejasz do wskazanej komórki.

**2. Analiza materiału:** dopiero działający notebook wysyła prompty i
ograniczony pakiet fragmentów przez API. W formularzu możesz wybrać
`gemini` albo `openai`; dalsze komórki pozostają takie same.

Dla wybranego providera dodaj w Colab Secrets tylko odpowiedni klucz:
`GEMINI_API_KEY` albo `OPENAI_API_KEY`. Przy OpenAI pole
`OPENAI_STORE` pozostaje jawną decyzją uczestnika; adapter zapisze jego
wartość i identyfikator odpowiedzi w lokalnym logu przebiegu.

Tryb `mock` sprawdza przepływ bez wysyłania danych. Czat Colaba służy do
vibe codingu, a `analysis_api` wyłącznie do porównań analitycznych.


In [ ]:
# @title Infrastruktura warsztatu — uruchom bez edycji { display-mode: "form" }
%pip install -q pandas "google-genai>=2.0.0" "openai>=2.0.0"

from pathlib import Path
import base64
import json
import os
import re
import subprocess
import sys
import pandas as pd
from IPython.display import display

PUBLIC_REPOSITORY_URL = "https://github.com/caqdastm/ai_qda-workshop-1u.git"
PUBLIC_REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u"
REPOSITORY_REF = "main" # @param {type:"string"}

def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

PARTICIPANT_REPOSITORY = str(
    _colab_secret("AI_QDA_REPOSITORY") or ""
).strip().rstrip("/")
if PARTICIPANT_REPOSITORY.endswith(".git"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY[:-4]
if PARTICIPANT_REPOSITORY.startswith("https://github.com/"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY.removeprefix(
        "https://github.com/"
    )
if PARTICIPANT_REPOSITORY and not re.fullmatch(
    r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", PARTICIPANT_REPOSITORY
):
    raise ValueError(
        "Sekret AI_QDA_REPOSITORY podaj jako login/nazwa-repozytorium."
    )
if PARTICIPANT_REPOSITORY.lower() == PUBLIC_REPOSITORY_SLUG.lower():
    raise ValueError(
        "AI_QDA_REPOSITORY musi wskazywać Twoje prywatne repo, "
        "nie repo prowadzących."
    )

GITHUB_TOKEN = _colab_secret("GITHUB_TOKEN")
if PARTICIPANT_REPOSITORY:
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "Włącz dla tego notebooka dostęp do sekretu GITHUB_TOKEN."
        )
    REPOSITORY_URL = f"https://github.com/{PARTICIPANT_REPOSITORY}.git"
    WORKSPACE_MODE = "participant_repository"
else:
    REPOSITORY_URL = PUBLIC_REPOSITORY_URL
    WORKSPACE_MODE = "public_demo"

local_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (
        path for path in local_candidates
        if (path / "04_vibe_coding" / "workshop_support.py").is_file()
    ),
    Path("/content/ai_qda_workshop_workspace"),
)
if not (REPO_ROOT / "04_vibe_coding" / "workshop_support.py").is_file():
    clone_environment = os.environ.copy()
    if GITHUB_TOKEN:
        encoded = base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
        ).decode("ascii")
        clone_environment["GIT_CONFIG_COUNT"] = "1"
        clone_environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
        clone_environment["GIT_CONFIG_VALUE_0"] = (
            f"AUTHORIZATION: basic {encoded}"
        )
    clone_result = subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ],
        env=clone_environment,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            "Nie udało się otworzyć repozytorium roboczego. Sprawdź "
            "sekrety AI_QDA_REPOSITORY i GITHUB_TOKEN oraz dostęp tokenu "
            f"do repo. Git: {clone_result.stderr.strip()}"
        )

%cd $REPO_ROOT
support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from workshop_support import (
    AnalysisAPI,
    load_dataframe,
    load_workshop_packet,
    publish_outputs_to_github,
    procedure_prompt,
    read_secret,
    save_dataframe,
    save_json,
)

WORKSPACE = REPO_ROOT / "04_vibe_coding" / "outputs"
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Tryb workspace:", WORKSPACE_MODE)
print("Repo uczestnika:", PARTICIPANT_REPOSITORY or "brak — publiczny tryb demonstracyjny")
print("Katalog przekazania między blokami:", WORKSPACE)


In [ ]:
# @title API analityczne — zmiana providera nie zmienia dalszych komórek { display-mode: "form" }
PROVIDER = "mock" # @param ["mock", "gemini", "openai"]
GEMINI_MODEL = "gemini-3.6-flash" # @param {type:"string"}
OPENAI_MODEL = "gpt-5.4-mini" # @param {type:"string"}
OPENAI_STORE = True # @param {type:"boolean"}
AUTHORIZE_API_CALLS = False # @param {type:"boolean"}
MAX_API_CALLS = 2 # @param {type:"integer"}

analysis_api = AnalysisAPI(
    provider=PROVIDER,
    gemini_model=GEMINI_MODEL,
    openai_model=OPENAI_MODEL,
    openai_store=OPENAI_STORE,
    authorize_api_calls=AUTHORIZE_API_CALLS,
    max_api_calls=MAX_API_CALLS,
)
print("Provider:", PROVIDER, "| model:", analysis_api.model, "| limit:", MAX_API_CALLS)


In [ ]:
# @title Wczytaj D, F i mapowanie z poprzednich bloków { display-mode: "form" }
d_assignments = load_dataframe(WORKSPACE / "02_d_assignments.csv")
focused = load_dataframe(WORKSPACE / "03_focused_categories.csv")
d_to_f = load_dataframe(WORKSPACE / "03_d_to_f.csv")
if focused.empty:
    raise ValueError("Najpierw wróć do bloku 3 i zapisz co najmniej jedną kandydacką F.")
display(focused)
display(d_to_f)


In [ ]:
# @title Karta procedury operacjonalizacji książki { display-mode: "form" }
cel = "Z kilku zgodnych F utworzyć używalną przez drugiego kodera kartę S, a S umieścić w oszczędnym zakresie T." # @param {type:"string"}
wymagania_S = "Nazwa, definicja, kryterium włączenia i wyłączenia, granica oraz przykład prowadzący do źródła." # @param {type:"string"}
rola_T = "Porządkuje szerszy zakres analityczny i wspiera memo; nie jest bezpośrednim kodem fragmentu." # @param {type:"string"}
decyzja_badacza = "Granica S, wybór zakresu T, postępowanie z kontrprzykładem i ewentualny status accepted." # @param {type:"string"}
PROCEDURE_CARD = {
    "goal": cel,
    "input": "Kandydackie F, mapowanie D–F oraz cytaty i memo D.",
    "observable_result": "Kandydacka hierarchia T–S–F–D z pełną kartą S i śladem do tekstu.",
    "automatic_check": "Rodzice istnieją, nie ma cykli ani duplikatów, S ma wymagane pola, a D zachowuje cytat.",
    "researcher_decision": decyzja_badacza,
}


## Vibe coding w czacie Colaba: `check_codebook_hierarchy`

1. Uruchom następną komórkę, aby wyświetlić instrukcję.
2. Otwórz panel czatu AI w Colabie i wklej całą instrukcję.
3. Poproś najpierw o krótkie powtórzenie kontraktu zwykłym językiem,
   a następnie o jedną funkcję — bez przebudowy notebooka.
4. Wklej otrzymaną funkcję do komórki **KOMÓRKA UCZESTNIKA**.

Na tym etapie nie korzystasz z klucza API i nie prosisz API
analitycznego o napisanie kodu.


In [ ]:
# @title Wyświetl instrukcję dla czatu Colaba { display-mode: "form" }
appendix = """Napisz funkcję check_codebook_hierarchy(codebook).
Zwróć tabelę checklisty. Sprawdź unikalność code_id, istnienie
rodziców, brak rodzica T, relacje D→F→S→T oraz komplet pól S.
Nie oceniaj granic ani nie nadawaj accepted."""
FUNCTION_PROMPT = procedure_prompt(PROCEDURE_CARD, appendix)
print(FUNCTION_PROMPT)


In [ ]:
# KOMÓRKA UCZESTNIKA: wklej pełną funkcję otrzymaną od modelu.
check_codebook_hierarchy = None


## Analiza korpusu przez API

Teraz kod pomocniczy jest już w notebooku. Dwa kolejne wywołania API
dostają ten sam materiał, ale inaczej sformułowane zadania analityczne.
Porównujesz wpływ promptu, a nie SDK providera. Zmiana `gemini` na
`openai` odbywa się wyłącznie w formularzu **API analityczne**.


In [ ]:
# @title Dwa wywołania analityczne: ogólna synteza i operacyjna karta S { display-mode: "form" }
f_material = focused.to_csv(index=False)
prompt_a = f"""Połącz poniższe kategorie F w szersze kategorie i nadaj im nazwy.\n\n{f_material}"""
prompt_b = f"""Rozważ syntezę poniższych F w kandydacką kartę S.
Nie wymuszaj połączenia. Jeśli mechanizmy są zgodne, podaj definicję,
kryterium włączenia, kryterium wyłączenia, granicę, kontrprzykład i
uzasadnienie. Zaproponuj oszczędny T zgodny z rolą: {rola_T}
Nie nadawaj statusu accepted.\n\n{f_material}"""
response_a = analysis_api.run_analysis(prompt_a, task_label="04_general_synthesis")
response_b = analysis_api.run_analysis(prompt_b, task_label="04_operational_s_card")
display(pd.DataFrame([
    {"wariant": "A — ogólna synteza", "odpowiedź": response_a},
    {"wariant": "B — operacyjna karta", "odpowiedź": response_b},
]))


## Powrót do materiału i decyzja badacza

Odpowiedzi API są kandydackie. Wróć do cytatów i zapisz własną decyzję
w formularzu poniżej. Checklista może wykryć błąd struktury, ale nie
potwierdza trafności kodu, kategorii ani granicy interpretacji.


In [ ]:
# @title Zapisz decyzję o jednej S i jednym T { display-mode: "form" }
s_name = "" # @param {type:"string"}
s_definition = "" # @param {type:"string"}
s_include = "" # @param {type:"string"}
s_exclude = "" # @param {type:"string"}
s_boundary = "" # @param {type:"string"}
s_f_ids = "" # @param {type:"string"}
t_name = "" # @param {type:"string"}
t_scope = "" # @param {type:"string"}
review_decision = "needs_review" # @param ["candidate", "needs_review", "proposed_revision"]

selected_f = {value.strip() for value in s_f_ids.split(",") if value.strip()}
parent_by_d = d_to_f.set_index("code_id")["focused_id"].to_dict() if not d_to_f.empty else {}
rows = []
for row in d_assignments.itertuples():
    if str(row.code_name).strip():
        parent_id = parent_by_d.get(row.code_id, "")
        rows.append({
            "code_id": row.code_id, "code_level": "descriptive", "code_name": row.code_name,
            "parent_code_id": parent_id, "definition": "",
            "inclusion_criteria": "", "exclusion_criteria": "",
            "example_quote": row.evidence_quote,
            "text_unit_id": row.text_unit_id, "source_file": row.source_file,
            "boundary_condition": "",
            "review_status": row.review_status if parent_id else "needs_review",
        })
for row in focused.itertuples():
    parent_id = "S01" if row.focused_id in selected_f and s_name.strip() else ""
    rows.append({
        "code_id": row.focused_id, "code_level": "focused", "code_name": row.focused_name,
        "parent_code_id": parent_id, "definition": row.analytic_rationale,
        "inclusion_criteria": "", "exclusion_criteria": "",
        "example_quote": "", "text_unit_id": "", "source_file": "",
        "boundary_condition": row.boundary,
        "review_status": row.review_status if parent_id else "needs_review",
    })
if s_name.strip():
    rows.append({
        "code_id": "S01", "code_level": "synthetic", "code_name": s_name.strip(),
        "parent_code_id": "T01" if t_name.strip() else "", "definition": s_definition.strip(),
        "inclusion_criteria": s_include.strip(), "exclusion_criteria": s_exclude.strip(),
        "example_quote": "", "text_unit_id": "", "source_file": "",
        "boundary_condition": s_boundary.strip(), "review_status": review_decision,
    })
if t_name.strip():
    rows.append({
        "code_id": "T01", "code_level": "theoretical", "code_name": t_name.strip(),
        "parent_code_id": "", "definition": t_scope.strip(),
        "inclusion_criteria": "", "exclusion_criteria": "",
        "example_quote": "", "text_unit_id": "", "source_file": "",
        "boundary_condition": "", "review_status": review_decision,
    })
codebook = pd.DataFrame(rows)
display(codebook)


In [ ]:
# @title Rozwiązanie awaryjne i checklista hierarchii { display-mode: "form" }
def prepared_check_codebook_hierarchy(codebook):
    if codebook.empty:
        return pd.DataFrame([{"kontrola": "Książka zawiera decyzje", "zaliczona": False}])
    ids = set(codebook["code_id"])
    parents = set(codebook["parent_code_id"].astype(str)) - {""}
    s_rows = codebook.loc[codebook["code_level"].eq("synthetic")]
    expected_parent = {
        "descriptive": "focused",
        "focused": "synthetic",
        "synthetic": "theoretical",
    }
    level_by_id = codebook.set_index("code_id")["code_level"].to_dict()
    parent_levels_ok = all(
        row.code_level == "theoretical" or row.review_status == "needs_review" or
        (row.parent_code_id and level_by_id.get(row.parent_code_id) == expected_parent[row.code_level])
        for row in codebook.itertuples()
    )
    return pd.DataFrame([
        {"kontrola": "ID są unikalne", "zaliczona": codebook["code_id"].is_unique},
        {"kontrola": "Rodzice istnieją", "zaliczona": parents.issubset(ids)},
        {"kontrola": "Poziomy rodziców są zgodne", "zaliczona": parent_levels_ok},
        {"kontrola": "T nie ma rodzica", "zaliczona": codebook.loc[codebook["code_level"].eq("theoretical"), "parent_code_id"].astype(str).eq("").all()},
        {"kontrola": "S ma definicję i kryteria", "zaliczona": (not s_rows.empty) and s_rows[["definition", "inclusion_criteria", "exclusion_criteria"]].astype(str).apply(lambda col: col.str.strip().ne("").all()).all()},
        {"kontrola": "Brak accepted od modelu", "zaliczona": not codebook["review_status"].astype(str).eq("accepted").any()},
    ])
if not callable(globals().get("check_codebook_hierarchy")):
    check_codebook_hierarchy = prepared_check_codebook_hierarchy
checklist = check_codebook_hierarchy(codebook)
display(checklist)
print("Zielona checklista nie zatwierdza granic S ani zakresu T.")


In [ ]:
# @title Zapisz książkę kandydacką i handoff do Codexa { display-mode: "form" }
save_dataframe(WORKSPACE / "04_candidate_codebook.csv", codebook)
save_json(WORKSPACE / "04_procedure_card.json", PROCEDURE_CARD)
save_json(WORKSPACE / "04_handoff_to_codex.json", {
    "workspace": str(WORKSPACE),
    "research_model": "relevance → D → F → operational S → parsimonious T",
    "fragile_point": "uzupełnij po review",
    "one_needed_validation": "uzupełnij po review",
    "one_adaptation": "uzupełnij przed częścią Codex",
    "non_delegable": decyzja_badacza,
})
analysis_api.export_runs(WORKSPACE / "04_prompt_runs.jsonl")
print("Zapisano blok 4 w", WORKSPACE)


In [ ]:
# @title Zapisz trwały punkt kontrolny w swoim repo GitHub { display-mode: "form" }
PUBLISH_RESULTS_TO_GITHUB = False # @param {type:"boolean"}
INCLUDE_API_LOGS_IN_GITHUB = False # @param {type:"boolean"}

if PUBLISH_RESULTS_TO_GITHUB:
    if WORKSPACE_MODE != "participant_repository":
        raise RuntimeError(
            "Ten notebook działa na publicznym wzorcu. Dodaj w Colab "
            "Secrets AI_QDA_REPOSITORY i GITHUB_TOKEN, włącz ich "
            "dostęp i uruchom notebook ponownie od początku."
        )
    publication = publish_outputs_to_github(
        REPO_ROOT,
        [WORKSPACE],
        message="AI QDA: zakończ blok 04",
        participant_repository=PARTICIPANT_REPOSITORY,
        token=read_secret("GITHUB_TOKEN"),
        branch=REPOSITORY_REF,
        include_api_logs=INCLUDE_API_LOGS_IN_GITHUB,
    )
    print("Zapis GitHub:", publication["status"])
    print("Commit:", publication.get("commit", "bez nowej zmiany"))
    print("Pliki:", publication["paths"])
else:
    print(
        "Wyniki są teraz tylko w runtime. Po ich sprawdzeniu ustaw "
        "PUBLISH_RESULTS_TO_GITHUB=True i uruchom tę komórkę ponownie."
    )


## Przejście do Codexa

Po uzyskaniu statusu `pushed` sklonuj lokalnie własne prywatne repo,
nie czysty wzorzec prowadzących. Codex ma najpierw odtworzyć model
kodowania z czterech kart procedur i artefaktów zapisanych w
`04_vibe_coding/outputs/`. Dopiero potem proponuje jedną małą
implementację lub adaptację. Wynik full służy jako przykład skali,
nie kod do kopiowania.
